In [1]:
import json
from collections import defaultdict

In [2]:
entt_to_cat = defaultdict(set)
entt_redone = set()
original_set = set()
original_processed_set = set()

for i in range(10):
    with open(f"entity_output_files_redone/chapter_{i}_output.json", "r", encoding="utf-8") as f:
        file = json.load(f)
        for grp in file:
            for obj in grp['extracted_entities']:
                entt_redone.update(obj['entities'])

with open("processing_state.json", "r", encoding="utf-8") as f:
    ps_file = json.load(f)
    entt_dict = ps_file['global_entity_registry']
    for k, v_list in entt_dict.items():
        for entt in v_list:
            original_set.add(entt)

with open("entity_info.json", "r", encoding="utf-8") as f:
    info_file = json.load(f)
    for k, obj in info_file.items():
        original_processed_set.add(obj['name'])
        

assert(len(original_processed_set) == len(original_set))
        

In [12]:
print(len(entt_redone), len(original_processed_set))
print(len(entt_redone - original_processed_set))
print(len(original_processed_set - entt_redone))

1975 1192
1092
309


In [8]:
common_entities = original_processed_set-(entt_redone-original_processed_set)
for k, obj in info_file.items():
    entt_to_cat[obj['name']] = obj['types']

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
model = SentenceTransformer("krutrim-ai-labs/vyakyarth")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [13]:
sentences = [
    "दशमी तिथि", 
    "एकादशी", 
    "उपवास",
    "व्रत"
]
embeddings = np.array(model.encode(sentences))

print(cosine_similarity(embeddings[0].reshape(1, -1), embeddings[1].reshape(1, -1)))
print(cosine_similarity(embeddings[2].reshape(1, -1), embeddings[3].reshape(1, -1)))

[[0.5246358]]
[[0.6991596]]


In [9]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional

In [10]:
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

In [19]:
def call_llm(system_prompt, user_query, model_name="Qwen/Qwen2.5-14B-Instruct", max_tokens=8192, response_model=None):
    api_params = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": max_tokens,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content

In [13]:
import json
import numpy as np
from enum import Enum
from typing import List, Optional, Dict
from pydantic import BaseModel, Field, field_validator
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from openai import OpenAI

# ========================
# 1. ENUM FOR CONFIDENCE
# ========================
class ConfidenceLevel(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

# ========================
# 2. PYDANTIC MODEL (no float confidence, flat types allowed)
# ========================
class CategorizationResult(BaseModel):
    matched: bool = Field(description="True if new entity matches an existing entity semantically")
    matched_entity: Optional[str] = Field(None, description="Name of the original entity that matches (if matched=true)")
    types: List[str] = Field(default_factory=list, description="List of category names (flat list as stored in entity_info.json)")
    confidence: ConfidenceLevel = Field(description="Qualitative confidence: low, medium, high")
    reasoning: str = Field(description="Brief explanation")

    # Optional: validate that types are non-empty when matched=true
    @field_validator('types')
    @classmethod
    def types_not_empty_if_matched(cls, v, info):
        if info.data.get('matched') and not v:
            raise ValueError("If matched=true, types cannot be empty")
        return v

# ========================
# 3. LOAD ORIGINAL ENTITIES FROM entity_info.json
# ========================
def load_original_entities(entity_info_path: str) -> Dict[str, List[str]]:
    """
    Reads entity_info.json and returns a dict: entity_name -> list of types (flat).
    """
    with open(entity_info_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    mapping = {}
    for node_id, obj in data.items():
        name = obj.get("name")
        types = obj.get("types", [])
        if name:   # skip if name missing
            mapping[name] = types
    return mapping

# Load
entity_info_path = "entity_info.json"   # adjust path if needed
original_mapping = load_original_entities(entity_info_path)
original_entities = list(original_mapping.keys())
original_types = list(original_mapping.values())   # list of lists (flat)

print(f"Loaded {len(original_entities)} original entities with their types.")

# ========================
# 4. GET NEW ENTITIES (from your redone outputs)
# ========================
# Simulate or reuse your existing set. We'll assume your `entt_redone` is already computed.
# For completeness, here's how you might compute it (adjust paths as needed):
entt_redone = set()
for i in range(10):
    with open(f"entity_output_files_redone/chapter_{i}_output.json", "r", encoding="utf-8") as f:
        file = json.load(f)
        for grp in file:
            for obj in grp['extracted_entities']:
                # obj['entities'] is a list of entity strings
                entt_redone.update(obj['entities'])

original_names_set = set(original_entities)
new_entities = entt_redone - original_names_set
print(f"Found {len(new_entities)} new entities to categorize.")

# ========================
# 5. EMBEDDINGS & LLM SETUP
# ========================
EMBEDDING_MODEL = "sentence-transformers/LaBSE"
LLM_BASE_URL = "http://127.0.0.1:8001/v1"
LLM_API_KEY = "none"
LLM_MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
TOP_K = 20
CONFIDENCE_THRESHOLD = ConfidenceLevel.MEDIUM   # we'll treat 'medium' and 'high' as confident enough
TEMPERATURE = 0.2
MAX_TOKENS = 4096

model = SentenceTransformer(EMBEDDING_MODEL)
print("Encoding original entities...")
orig_embeddings = model.encode(original_entities, show_progress_bar=True)
orig_embeddings = normalize(orig_embeddings)

client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

# ========================
# 6. PROMPT (inform LLM about confidence strings)
# ========================
SYSTEM_PROMPT = """You are an expert in classical Hindi and Sanskrit religious texts.
Your task is to compare a new entity with a list of already categorized entities and decide if the new entity is essentially the same as (or a specific instance/subtype of) any of them.

The existing entities have been manually categorized. Their types are stored as a flat list of category names (e.g., ["special_date","date","Time","Activity","festival","Event"]). You should copy the entire list from the matched entity.

Rules:
- If the new entity matches an existing entity semantically (even if spelling differs), set matched=true, provide matched_entity, and copy the types list exactly.
- If no match, set matched=false, types empty.
- confidence must be one of: "low", "medium", "high". Use "high" for clear synonyms (e.g., उपवास/व्रत), "medium" for partial similarity, "low" when unsure.

Output only valid JSON according to the schema. Do not add extra text."""

USER_PROMPT_TEMPLATE = """New entity: "{new_entity}"

Here are the top {top_k} most similar already‑categorized entities (by vector similarity):

{top_entities_list}

Based on your knowledge, determine if "{new_entity}" is semantically equivalent to or a subtype of any of the above.
Return a JSON object with fields: matched, matched_entity (if matched=true), types (copy from matched entity), confidence ("low"/"medium"/"high"), reasoning."""

def format_top_entities(top_list):
    lines = []
    for i, (name, types, score) in enumerate(top_list, 1):
        type_str = ", ".join(types) if types else "(no types)"
        lines.append(f"{i}. {name} (cosine={score:.3f}) → types: [{type_str}]")
    return "\n".join(lines)

# ========================
# 7. MAIN LOOP (qualitative confidence)
# ========================
resolved_results: Dict[str, List[str]] = {}   # new_entity -> types
unresolved_list: List[str] = []               # entities that need manual review

for idx, new_ent in enumerate(new_entities, 1):
    print(f"[{idx}/{len(new_entities)}] {new_ent}")

    # Retrieve top K similar
    new_emb = model.encode([new_ent])
    new_emb = normalize(new_emb)
    sim = cosine_similarity(new_emb, orig_embeddings)[0]
    top_idx = np.argsort(sim)[-TOP_K:][::-1]
    top_entities = [(original_entities[i], original_types[i], sim[i]) for i in top_idx]

    user_prompt = USER_PROMPT_TEMPLATE.format(
        new_entity=new_ent,
        top_k=TOP_K,
        top_entities_list=format_top_entities(top_entities)
    )

    # Call LLM
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS
        )
        content = response.choices[0].message.content
        result = CategorizationResult.model_validate_json(content)
    except Exception as e:
        print(f"  ❌ Error: {e}\n  Raw output: {content if 'content' in locals() else 'No response'}")
        unresolved_list.append(new_ent)
        continue

    # Decide: accept if matched and confidence is at least "medium"
    if result.matched and result.confidence in [ConfidenceLevel.HIGH]:
        resolved_results[new_ent] = result.types
        print(f"  ✅ Matched to '{result.matched_entity}' with types {result.types} (conf={result.confidence.value})")
    else:
        unresolved_list.append(new_ent)
        print(f"  ❌ Unresolved: matched={result.matched}, conf={result.confidence.value}, reason: {result.reasoning[:80]}")

# ========================
# 8. SAVE OUTPUTS (separate files, not modifying entity_info.json)
# ========================
with open("auto_categorized.json", "w", encoding="utf-8") as f:
    json.dump(resolved_results, f, ensure_ascii=False, indent=2)

with open("unresolved_entities.json", "w", encoding="utf-8") as f:
    json.dump(unresolved_list, f, ensure_ascii=False, indent=2)

print(f"\n✅ Done. Auto‑categorized: {len(resolved_results)}. Unresolved (manual): {len(unresolved_list)}.")

Loaded 1192 original entities with their types.
Found 1092 new entities to categorize.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding original entities...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

[1/1092] आशीर्वाद
  ❌ Unresolved: matched=False, conf=low, reason: None of the provided entities are exact synonyms or subtypes of 'आशीर्वाद'. Whil
[2/1092] कालावयवों
  ✅ Matched to 'कालावयव' with types ['measurement_unit', 'Time', 'measure_quantity', 'Concept'] (conf=high)
[3/1092] द्वादशी व्रत
  ✅ Matched to 'सहस्र द्वादशी व्रत' with types ['festival', 'Activity'] (conf=high)
[4/1092] श्री
  ❌ Unresolved: matched=False, conf=low, reason: The term 'श्री' is a respectful honorific title often used before the names of d
[5/1092] जन्म-महोत्सव
  ✅ Matched to 'महोत्सव' with types ['festival', 'Activity'] (conf=high)
[6/1092] श्रवणमास
  ❌ Unresolved: matched=True, conf=medium, reason: श्रवणमास refers to a lunar month dedicated to hearing sacred stories or listenin
[7/1092] विद्वान व्यक्ति
  ✅ Matched to 'विद्वान' with types ['attribute', 'Social_Group_&_Role', 'Concept'] (conf=high)
[8/1092] वसोर्धारा
  ❌ Unresolved: matched=False, conf=low, reason: The term 'वसोर्धारा' does not clearly mat

In [ ]:
class ExtractedEntities(BaseModel):
    # 🔥 CRITICAL: The scratchpad MUST be the first field.
    step_by_step_reasoning: str = Field(
        description="Think step-by-step. Read the sentence, check the formal definitions, and deduce which entities belong to which category before listing them."
    )
    
    Mythical_Entity: Optional[List[str]] = Field(default=None)
    Celestial_Entity: Optional[List[str]] = Field(default=None)
    Phenomenon: Optional[List[str]] = Field(default=None)
    Time: Optional[List[str]] = Field(default=None)
    Food: Optional[List[str]] = Field(default=None)
    Activity: Optional[List[str]] = Field(default=None)
    Concept: Optional[List[str]] = Field(default=None)
    Object: Optional[List[str]] = Field(default=None)
    Living_Being: Optional[List[str]] = Field(default=None)
    Text: Optional[List[str]] = Field(default=None)
    Location: Optional[List[str]] = Field(default=None)
    Primordial_Element: Optional[List[str]] = Field(default=None)
    Medical_Concept: Optional[List[str]] = Field(default=None)
    Geographical_Feature: Optional[List[str]] = Field(default=None)
    Event: Optional[List[str]] = Field(default=None)
    Emotions: Optional[List[str]] = Field(default=None)
    Body_part: Optional[List[str]] = Field(default=None)
    Sanskrit_text: Optional[List[str]] = Field(default=None)
    Other: Optional[List[str]] = Field(default=None)

    Social_Group_and_Role: Optional[List[str]] = Field(
        default=None,
        alias="Social_Group_&_Role"
    )

    model_config = {"populate_by_name": True}

In [23]:
with open("prompts/recategorization_lvl_1.txt", 'r', encoding='utf-8') as f:
    CATEGORIZATION_PROMPT = f.read()

In [45]:
unresolved_set = set(unresolved_list)
resolved_entities = defaultdict(set)

for i in range(10):
    with open(f"entity_output_files_redone/chapter_{i}_output.json", 'r', encoding='utf-8') as f:
        entt_file = json.load(f)
        for grp in entt_file:
            for obj in grp['extracted_entities']:
                sentence = obj['sentence']
                # Only entities still unresolved
                entities = [entt for entt in obj['entities'] if entt in unresolved_set]
                if not entities:
                    continue

                for entt in entities:
                    user_query = f"\n\nsentence: {sentence}\nentities: [{entt}]\n\n"
                    success = False

                    # Retry up to 3 times
                    for attempt in range(1, 4):
                        try:
                            response = call_llm(
                                CATEGORIZATION_PROMPT,
                                user_query,
                                max_tokens=4096,
                                response_model=ExtractedEntities
                            )
                            parsed_response = ExtractedEntities.model_validate_json(response).model_dump()

                            # Process the response
                            for k, v_list in parsed_response.items():
                                if k == 'step_by_step_reasoning':
                                    continue
                                if not v_list:
                                    continue
                                for resolved_entt in v_list:
                                    if resolved_entt not in unresolved_set:
                                        continue
                                    resolved_entities[resolved_entt].add(k)
                                    # Remove from unresolved_set immediately
                                    unresolved_set.discard(resolved_entt)

                            success = True
                            print(user_query)
                            print(response)
                            print("=" * 68)
                            break  # Exit retry loop on success

                        except Exception as e:
                            print(f"Attempt {attempt} failed for entity '{entt}': {e}")
                            if attempt == 3:
                                print(f"Giving up on '{entt}' after 3 attempts.")
                            else:
                                print(f"Retrying... ({attempt}/3)")

                    if not success:
                        # Entity remains in unresolved_set; do nothing else
                        print(f"Failed to categorize '{entt}' after 3 attempts.")
                        print("=" * 68)



sentence: चैत्र शुक्ल प्रतिपदा के दिन मिश्री तथा काली मिर्च सहित नीम के पत्ते खाने चाहिए।
entities: [नीम के पत्ते]


{
  "step_by_step_reasoning": "नीम के पत्ते is a leaf of the neem tree, which is consumed as food.",
  "Food": ["नीम के पत्ते"],
  "Activity": ["चैत्र शुक्ल प्रतिपदा"],
  "Concept": ["मिश्री", "काली मिर्च"]
}


sentence: मृत प्रकृति को जीवन प्रदान करने के कारण ही सूर्य, जो उष्णता का आकर है, 'मार्तण्ड' कहा जाता है।
entities: [मृत प्रकृति]


{
  "step_by_step_reasoning": "मृत प्रकृति is described as a state or condition of nature, fitting under the Concept category.",
  "Concept": ["मृत प्रकृति"]
}


sentence: प्रतिपदा का दिन चन्द्रमा की प्रथम कला के आरम्भ का दिन है।
entities: [प्रथम कला]


{
  "step_by_step_reasoning": "प्रथम कला refers to the first phase or crescent of the moon, which is a phenomenon related to the moon's appearance.",
  "Phenomenon": ["प्रथम कला"]
}


sentence: यदि किसी दूसरे दिन वर्षारम्भ माना जाता, तो उस समय चन्द्रमा नवजात या पूर्ण होता, जो आरम्भ का

In [54]:
base_classes = [
    "Mythical_Entity",
    "Celestial_Entity",
    "Phenomenon",
    "Time",
    "Food",
    "Activity",
    "Concept",
    "Object",
    "Living_Being",
    "Text",
    "Location",
    "Primordial_Element",
    "Medical_Concept",
    "Geographical_Feature",
    "Event",
    "Emotions",
    "Body_part",
    "Sanskrit_text",
    "Other",
    "Social_Group_&_Role"
]

In [63]:
cat_to_entt = defaultdict(set)

In [65]:
for k, v_list in resolved_results.items():
    if all(cat in base_classes for cat in v_list):
        for cat in v_list:
            cat_to_entt[cat].add(k)
            

In [66]:
cat_to_entt

defaultdict(set,
            {'Body_part': {'उदर',
              'कंठ के रोग',
              'कान',
              'चरणों',
              'नसों',
              'नेत्रों',
              'पैरों',
              'प्लीहा',
              'मृत शरीर',
              'स्तन',
              'स्नायुओं',
              'स्पर्श'},
             'Object': {'अक्षत',
              'अनुलेपन',
              'आधिभौतिक गंगाजी',
              'उत्तम वस्तु',
              'उत्तम वस्तुओं',
              'उपभोग्य वस्तुओं',
              'ऋतु के अनुकूल वस्तु',
              'औषधों',
              'कंबल',
              'कमल',
              'कमल के फूल',
              'करवीर',
              'कलश',
              'कुम्भा',
              'कुश',
              'कुशजल',
              'कुशा का जल',
              'कोमल पत्ते',
              'गंगाजल',
              'गङ्गाजी के जल',
              'गुणकारी वस्तुएँ',
              'गोता',
              'गोमय',
              'घृत',
              'चाँदी का पलंग',
              'चा

In [69]:
print({k: list(v_set) for k, v_set in cat_to_entt.items()})

{'Body_part': ['प्लीहा', 'उदर', 'चरणों', 'कान', 'मृत शरीर', 'पैरों', 'स्नायुओं', 'नसों', 'स्पर्श', 'स्तन', 'कंठ के रोग', 'नेत्रों'], 'Object': ['वस्तुओं', 'नवीन वस्तु', 'गोता', 'शस्त्र', 'मुर्दे', 'स्नेहों', 'सुवर्ण के भार से बनाया गया नाग', 'पुष्प-मालाएँ', 'सुगन्धित पुष्प', 'मिट्टी का नाग', 'ऋतु के अनुकूल वस्तु', 'भूषण', 'वेदी', 'कलश', 'कमल के फूल', 'पत्र-पुष्पों', 'नाग की प्रतिमा', 'पूरने', 'सूत्र', 'भस्म', 'पाँच वस्तुओं', 'श्रीराम की प्रतिमा', 'बरतन', 'पवित्र वस्त्र', 'पुष्पित', 'यज्ञोपवीत', 'उत्तम वस्तुओं', 'मृत्तिका की प्रतिमा', 'कुशजल', 'पीतल की प्रतिमा', 'कुम्भा', 'फल-पुष्पों', 'जल के घड़े', 'कंबल', 'कुशा का जल', 'सूतिका-गृह', 'स्फटिक की प्रतिमा', 'मुसल', 'शतपत्र', 'चाँदी का पलंग', 'गोमय', 'ताँबे', 'मणि की प्रतिमा', 'पट्टे', 'ढाल', 'मृत्तिका', 'शुद्ध गङ्गाजल', 'गुणकारी वस्तुएँ', 'रस्सी', 'पांच वस्तुएँ', 'झाँझ', 'अक्षत', 'चित्र-लिखित प्रतिमा', 'तन्तु', 'उपभोग्य वस्तुओं', 'मंगल कलश', 'दीप', 'चाँदी की प्रतिमा', 'मूर्ति', 'पुष्पों', 'कुश', 'ताँबे की प्रतिमा', 'तलवार', 'तप्तसुवर्ण', 

In [85]:
import json

FILE = "cat_lvl_0.json"
CATEGORY = "Mythical_Entity"

# category mapping
cat_dict = {
    0: 'Time',
    1: 'Activity',
    2: 'Location',
    3: 'Other',
    4: 'Food',
    5: 'Object',
    6: 'Text',
    7: 'Medical_Concept',
    8: 'Sanskrit_text',
    9: 'Social_Group_&_Role',
    10: 'Phenomenon',
    11: 'Concept',
    12: 'Event',
    13: 'Primordial_Element',
    14: 'Geographical_Feature',
    15: 'Emotions',
    16: 'Body_part',
    17: 'Living_Being',
    18: 'Celestial_Entity',
    19: 'Mythical_Entity'
}


# ----------------------------
# Load
# ----------------------------
with open(FILE, "r", encoding="utf-8") as f:
    data = json.load(f)
    
category_list = data[CATEGORY]

print(f"Total entities in {CATEGORY}: {len(category_list)}\n")


# ----------------------------
# Interactive loop
# ----------------------------
for idx, entity in enumerate(category_list[:]):  # copy to avoid iteration issues

    print("\n----------------------------------")
    print(f"[{idx+1}/{len(category_list)}] Entity: {entity}")
    print(f"Other categories: {resolved_results.get(entity)}")
    print("Categories:")

    for k, v in cat_dict.items():
        print(f"{k}: {v}")

    user_input = input(
        "\nEnter category numbers (space separated), "
        "'s' = skip, 'x' = remove from current: "
    ).strip()

    # ----------------------------
    # Skip (keep as is)
    # ----------------------------
    if user_input.lower() == "s":
        print("→ kept in same category")
        continue

    # ----------------------------
    # Remove from current category
    # ----------------------------
    if user_input.lower() == "x":
        data[CATEGORY] = [
            e for e in data[CATEGORY] if e != entity
        ]

        with open(FILE, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        print("→ removed from current category")
        continue

    # ----------------------------
    # Parse category input
    # ----------------------------
    try:
        selected = list(map(int, user_input.split()))
    except:
        print("Invalid input, skipping...")
        continue

    # ----------------------------
    # Remove from current category
    # ----------------------------
    data[CATEGORY] = [
        e for e in data[CATEGORY] if e != entity
    ]
    
    # ----------------------------
    # Assign to selected categories
    # ----------------------------
    for cat_id in selected:
        cat_name = cat_dict[cat_id]

        if cat_name not in data:
            data[cat_name] = []

        if entity not in data[cat_name]:
            data[cat_name].append(entity)


    # ----------------------------
    # Save immediately
    # ----------------------------
    with open(FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print("→ reassigned and saved")


print("\nDONE.")

Total entities in Mythical_Entity: 39


----------------------------------
[1/39] Entity: श्री
Other categories: ['Mythical_Entity']
Categories:
0: Time
1: Activity
2: Location
3: Other
4: Food
5: Object
6: Text
7: Medical_Concept
8: Sanskrit_text
9: Social_Group_&_Role
10: Phenomenon
11: Concept
12: Event
13: Primordial_Element
14: Geographical_Feature
15: Emotions
16: Body_part
17: Living_Being
18: Celestial_Entity
19: Mythical_Entity
→ kept in same category

----------------------------------
[2/39] Entity: नारायण
Other categories: ['Mythical_Entity']
Categories:
0: Time
1: Activity
2: Location
3: Other
4: Food
5: Object
6: Text
7: Medical_Concept
8: Sanskrit_text
9: Social_Group_&_Role
10: Phenomenon
11: Concept
12: Event
13: Primordial_Element
14: Geographical_Feature
15: Emotions
16: Body_part
17: Living_Being
18: Celestial_Entity
19: Mythical_Entity
→ kept in same category

----------------------------------
[3/39] Entity: वृत्र
Other categories: ['Mythical_Entity']
Categories:


In [4]:
INPUT_FILE = "cat_lvl_0.json"
ENTITY_FILE = "entity_info.json"
ENTITY_MAP_FILE = "entity_id_dict.json"

In [6]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

# ============================================================
# TIME TREE
# ============================================================
TIME_LEAF_NODES = [
    "day",
    "special_date",
    "generic_date",
    "month",
    "moon_phase",
    "measurement_unit",
    "measurement_method",
    "nakshatra",
    "other"
]

PARENT_MAP = {
    "day": ["Time", "day"],
    "special_date": ["Time", "date", "special_date"],
    "generic_date": ["Time", "date", "generic_date"],
    "month": ["Time", "month"],
    "moon_phase": ["Time", "moon_phase"],
    "measurement_unit": ["Time", "measurement_unit"],
    "measurement_method": ["Time", "measurement_method"],
    "nakshatra": ["Time", "nakshatra"],
    "other": ["Time", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/time.txt", 'r', encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class TimeClassification(BaseModel):
    scratchpad: str = Field(
        description="step-by-step deduction of which of the leaf_categories the entity belongs to."
    )
    leaf_category: Literal[
        "day",
        "special_date",
        "generic_date",
        "month",
        "moon_phase",
        "measurement_unit",
        "measurement_method",
        "nakshatra",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 256,
        "temperature": 0.1,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=TimeClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in TIME_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

time_entities = file["Time"]

print(f"Total Time entities: {len(time_entities)}")

# ============================================================
# LOAD OUTPUT FILES
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in time_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Time"]
        }

        entities[node_id] = entity

    # ----------------------------
    # skip if already classified
    # ----------------------------
    existing_types = set(entity.get("types", []))
    if len(existing_types) > 1:
        print("→ already classified")
        continue

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)

    print(f"→ {leaf}")

    # ----------------------------
    # update types using full path
    # ----------------------------
    full_path = PARENT_MAP[leaf]

    types = set(entity.get("types", []))
    types.update(full_path)

    entity["types"] = list(types)

    # ----------------------------
    # save immediately
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

print("\nDONE.")

Total Time entities: 20

Processing: श्रवणमास
→ month

Processing: पौष
→ month

Processing: चैत्रमास नवमी
→ generic_date

Processing: श्रवणनदात्रे
→ special_date

Processing: पुण्यकाल
→ other

Processing: भाद्रपद मास के शुक्लपक्ष की पंचमी तिथि
→ generic_date

Processing: जया
→ other

Processing: ज्येष्ठमास
→ month

Processing: कला-काष्ठा
→ measurement_unit

Processing: देशकाल
→ other

Processing: श्रावण शुक्ल पंचमी
→ generic_date

Processing: भाद्रपद की पंचमी
→ generic_date

Processing: प्रथम दिन
→ generic_date

Processing: श्रावण की पंचमी
→ generic_date

Processing: षष्ठी
→ generic_date

Processing: पूर्वदिन
→ other

Processing: आसुरी वेला
→ other

Processing: अंधकारयुक्त दिन
→ other

Processing: व्यतीपातयोग
→ other

Processing: प्रथम कला
→ measurement_unit

DONE.


In [7]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

# ============================================================
# ACTIVITY TREE
# ============================================================
ACTIVITY_LEAF_NODES = [
    "festival",
    "ritual_activity",
    "mundane_activity",
    "other"
]

PARENT_MAP = {
    "festival": ["Activity", "festival"],
    "ritual_activity": ["Activity", "ritual_activity"],
    "mundane_activity": ["Activity", "mundane_activity"],
    "other": ["Activity", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/activity.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class ActivityClassification(BaseModel):
    scratchpad: str = Field(
        description="step-by-step deduction of which of the leaf_categories the entity belongs to."
    )
    leaf_category: Literal[
        "festival",
        "ritual_activity",
        "mundane_activity",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=ActivityClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in ACTIVITY_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

activity_entities = file.get("Activity", [])

print(f"Total Activity entities: {len(activity_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in activity_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Activity"]
        }

        entities[node_id] = entity

    # ----------------------------
    # skip if already has Activity subtype
    # ----------------------------
    existing_types = set(entity.get("types", []))
    if any(t in ACTIVITY_LEAF_NODES for t in existing_types):
        print("→ already classified")
        continue

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)

    print(f"→ {leaf}")

    # ----------------------------
    # update types (full path)
    # ----------------------------
    full_path = PARENT_MAP[leaf]

    types = set(entity.get("types", []))
    types.update(full_path)

    entity["types"] = list(types)

    # ----------------------------
    # save immediately
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

print("\nDONE.")

Total Activity entities: 60

Processing: मंगल कार्य
→ ritual_activity

Processing: सर्वपापनाशक व्रत
→ ritual_activity

Processing: वर्धापन
→ ritual_activity

Processing: वसोर्धारा
→ other

Processing: युद्ध
→ other

Processing: उपाकर्म
→ ritual_activity

Processing: छिड़काव
→ mundane_activity

Processing: पारण
→ festival

Processing: पंचमी के व्रत
→ ritual_activity

Processing: खेलना
→ mundane_activity

Processing: वेदाध्ययन
→ ritual_activity

Processing: स्वयं भोजन
→ mundane_activity

Processing: तर्पण
→ ritual_activity

Processing: राख में हवन
→ ritual_activity

Processing: समागम
→ other

Processing: रक्षा-बन्धन
→ festival

Processing: बाँधने
→ mundane_activity

Processing: जलदान
→ ritual_activity

Processing: स्वाहाकार
→ ritual_activity

Processing: छिड़कना
→ mundane_activity

Processing: बैठाकर
→ mundane_activity

Processing: भक्तिपूर्वक पुस्तक-पाठ
→ ritual_activity

Processing: करे
→ other

Processing: राजसूय
→ ritual_activity

Processing: अष्टविध
→ other

Processing: दशविध
→ ritu

In [8]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

# ============================================================
# PHENOMENON TREE
# ============================================================
PHENOMENON_LEAF_NODES = [
    "celestial_phenomenon",
    "season",
    "natural_phenomenon",
    "other"
]

PARENT_MAP = {
    "celestial_phenomenon": ["Phenomenon", "celestial_phenomenon"],
    "season": ["Phenomenon", "season"],
    "natural_phenomenon": ["Phenomenon", "natural_phenomenon"],
    "other": ["Phenomenon", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/phenomenon.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class PhenomenonClassification(BaseModel):
    scratchpad: str = Field(
        description="step-by-step deduction of which of the leaf_categories the entity belongs to."
    )
    leaf_category: Literal[
        "celestial_phenomenon",
        "season",
        "natural_phenomenon",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=PhenomenonClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in PHENOMENON_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

phenomenon_entities = file.get("Phenomenon", [])

print(f"Total Phenomenon entities: {len(phenomenon_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in phenomenon_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Phenomenon"]
        }

        entities[node_id] = entity

    # ----------------------------
    # classify (always run)
    # ----------------------------
    leaf = classify_entity(entity_name)

    print(f"→ {leaf}")

    full_path = PARENT_MAP[leaf]

    existing_types = set(entity.get("types", []))

    # ----------------------------
    # check if new type already exists
    # ----------------------------
    if set(full_path).issubset(existing_types):
        print("→ no new type to add")
        continue

    # ----------------------------
    # add new types
    # ----------------------------
    existing_types.update(full_path)
    entity["types"] = list(existing_types)

    # ----------------------------
    # save only when modified
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

    print("→ updated")

Total Phenomenon entities: 1

Processing: बादल
→ natural_phenomenon
→ updated


In [9]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')


# ============================================================
# MEDICAL TREE
# ============================================================
MEDICAL_LEAF_NODES = [
    "disease",
    "symptom_physical",
    "symptom_mental",
    "secretion_internal",
    "secretion_external",
    "remedy",
    "other"
]

PARENT_MAP = {
    "disease": ["Medical_Concept", "disease"],
    "symptom_physical": ["Medical_Concept", "symptom_physical"],
    "symptom_mental": ["Medical_Concept", "symptom_mental"],
    "secretion_internal": ["Medical_Concept", "secretion_internal"],
    "secretion_external": ["Medical_Concept", "secretion_external"],
    "remedy": ["Medical_Concept", "remedy"],
    "other": ["Medical_Concept", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/medical_concept.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class MedicalClassification(BaseModel):
    scratchpad: str = Field(
        description="step-by-step deduction of which of the leaf_categories the entity belongs to."
    )
    leaf_category: Literal[
        "disease",
        "symptom_physical",
        "symptom_mental",
        "secretion_internal",
        "secretion_external",
        "remedy",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=MedicalClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in MEDICAL_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

medical_entities = file.get("Medical_Concept", [])

print(f"Total Medical entities: {len(medical_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in medical_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Medical_Concept"]
        }

        entities[node_id] = entity

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)

    print(f"→ {leaf}")

    full_path = PARENT_MAP[leaf]

    existing_types = set(entity.get("types", []))

    # ----------------------------
    # only add if new
    # ----------------------------
    if set(full_path).issubset(existing_types):
        print("→ no new type to add")
        continue

    existing_types.update(full_path)
    entity["types"] = list(existing_types)

    # ----------------------------
    # save
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

    print("→ updated")

print("\nDONE.")

Total Medical entities: 40

Processing: लाल गाय का गोमूत्र
→ secretion_external
→ updated

Processing: पश्चगव्य
→ remedy
→ updated

Processing: दद्मा
→ symptom_physical
→ updated

Processing: उद्र
→ symptom_physical
→ updated

Processing: अफारा
→ symptom_physical
→ updated

Processing: अतिसार
→ disease
→ updated

Processing: पैरों का सुन्न होना
→ symptom_physical
→ updated

Processing: पीड़ित प्राणियों
→ other
→ updated

Processing: शरीर पुष्ट
→ other
→ updated

Processing: आमवात
→ disease
→ updated

Processing: परस्त्री सेवन
→ other
→ updated

Processing: रोगाणु
→ other
→ updated

Processing: वायु की पीड़ा
→ symptom_physical
→ updated

Processing: बवासीर
→ disease
→ updated

Processing: आरोग्य
→ other
→ updated

Processing: कोढ़
→ symptom_physical
→ updated

Processing: चर्म-सम्बन्धी मलिनता
→ symptom_physical
→ updated

Processing: हैजा
→ disease
→ updated

Processing: रक्त की कमी
→ disease
→ updated

Processing: गुप्त पाप
→ other
→ updated

Processing: शारीरिक लाभ
→ other
→ updated



In [10]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

# ============================================================
# MYTHICAL TREE
# ============================================================
MYTHICAL_LEAF_NODES = [
    "metaphysical_entity",
    "deity",
    "avatar",
    "being_class",
    "individual_figure",
    "mythical_creature_object",
    "other"
]

PARENT_MAP = {
    "metaphysical_entity": ["Mythical_Entity", "metaphysical_entity"],
    "deity": ["Mythical_Entity", "deity"],
    "avatar": ["Mythical_Entity", "avatar"],
    "being_class": ["Mythical_Entity", "being_class"],
    "individual_figure": ["Mythical_Entity", "individual_figure"],
    "mythical_creature_object": ["Mythical_Entity", "mythical_creature_object"],
    "other": ["Mythical_Entity", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/mythical_entity.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class MythicalClassification(BaseModel):
    scratchpad: str = Field(
        description="step-by-step deduction of which of the leaf_categories the entity belongs to."
    )
    leaf_category: Literal[
        "metaphysical_entity",
        "deity",
        "avatar",
        "being_class",
        "individual_figure",
        "mythical_creature_object",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=MythicalClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in MYTHICAL_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

mythical_entities = file.get("Mythical_Entity", [])

print(f"Total Mythical entities: {len(mythical_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in mythical_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Mythical_Entity"]
        }

        entities[node_id] = entity

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)
    print(f"→ {leaf}")

    full_path = PARENT_MAP[leaf]
    existing_types = set(entity.get("types", []))

    # ----------------------------
    # only update if new info
    # ----------------------------
    if set(full_path).issubset(existing_types):
        print("→ no new type to add")
        continue

    existing_types.update(full_path)
    entity["types"] = list(existing_types)

    # ----------------------------
    # save
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

    print("→ updated")

print("\nDONE.")

Total Mythical entities: 26

Processing: श्री
→ other
→ updated

Processing: नारायण
→ deity
→ updated

Processing: वृत्र
→ individual_figure
→ updated

Processing: दक्ष प्रजापति
→ individual_figure
→ updated

Processing: गर्गाचार्य
→ individual_figure
→ updated

Processing: पुरन्दर
→ deity
→ updated

Processing: प्रभु
→ other
→ updated

Processing: वरुणरूप
→ avatar
→ updated

Processing: कंसासुर
→ individual_figure
→ updated

Processing: विद्याधर
→ being_class
→ updated

Processing: सर्वपापहारिणी भगवती भागीरथी
→ deity
→ updated

Processing: देवराज
→ deity
→ updated

Processing: सुरों
→ being_class
→ updated

Processing: इन्द्राणी
→ deity
→ updated

Processing: सूक्ष्म और भावगम्य देवता
→ deity
→ updated

Processing: अश्वतर
→ mythical_creature_object
→ updated

Processing: देवकी देवी
→ individual_figure
→ updated

Processing: धर्मनन्दन
→ individual_figure
→ updated

Processing: भगवती गङ्गाजी
→ deity
→ updated

Processing: इन्द्र
→ deity
→ updated

Processing: सर्पराज
→ mythical_creature_

In [11]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

# ============================================================
# LIVING BEING TREE
# ============================================================
LIVING_LEAF_NODES = [
    "human_generic",
    "human_individual",
    "animal",
    "plant",
    "mythical_living_being",
    "other"
]

PARENT_MAP = {
    "human_generic": ["Living_Being", "human_generic"],
    "human_individual": ["Living_Being", "human_individual"],
    "animal": ["Living_Being", "animal"],
    "plant": ["Living_Being", "plant"],
    "mythical_living_being": ["Living_Being", "mythical_living_being"],
    "other": ["Living_Being", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/living_being.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class LivingBeingClassification(BaseModel):
    scratchpad: str = Field(
        description="step-by-step deduction of which of the leaf_categories the entity belongs to."
    )
    leaf_category: Literal[
        "human_generic",
        "human_individual",
        "animal",
        "plant",
        "mythical_living_being",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=LivingBeingClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in LIVING_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

living_entities = file.get("Living_Being", [])

print(f"Total Living_Being entities: {len(living_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in living_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Living_Being"]
        }

        entities[node_id] = entity

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)
    print(f"→ {leaf}")

    full_path = PARENT_MAP[leaf]
    existing_types = set(entity.get("types", []))

    # ----------------------------
    # add only if new
    # ----------------------------
    if set(full_path).issubset(existing_types):
        print("→ no new type to add")
        continue

    existing_types.update(full_path)
    entity["types"] = list(existing_types)

    # ----------------------------
    # save
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

    print("→ updated")

print("\nDONE.")

Total Living_Being entities: 44

Processing: कंस
→ mythical_living_being
→ updated

Processing: याज्ञवल्क्य
→ human_individual
→ updated

Processing: कालिय
→ mythical_living_being
→ updated

Processing: सर्प
→ animal
→ updated

Processing: पद्म
→ plant
→ updated

Processing: ऊँट
→ animal
→ updated

Processing: बकरी
→ animal
→ updated

Processing: कर्कोटक
→ mythical_living_being
→ updated

Processing: शंका करनेवालों
→ human_generic
→ updated

Processing: माता देवकी
→ human_individual
→ updated

Processing: तत्कालप्रसूत श्रेष्ठ कन्या
→ human_generic
→ updated

Processing: कछुए
→ animal
→ updated

Processing: शंकरभट्ट
→ human_individual
→ updated

Processing: यशोदा
→ human_individual
→ updated

Processing: देवकी
→ human_individual
→ updated

Processing: धृतराष्ट्र
→ human_individual
→ updated

Processing: गधे
→ animal
→ updated

Processing: भैंस
→ animal
→ updated

Processing: दर्शक
→ human_generic
→ updated

Processing: पिंगल
→ human_individual
→ updated

Processing: शंखपाल
→ mythical_li

In [12]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

# ============================================================
# CONCEPT TREE
# ============================================================
CONCEPT_LEAF_NODES = [
    "abstract_concept",
    "attribute",
    "state",
    "action_process",
    "measure_quantity",
    "knowledge_linguistic",
    "other"
]

PARENT_MAP = {
    "abstract_concept": ["Concept", "abstract_concept"],
    "attribute": ["Concept", "attribute"],
    "state": ["Concept", "state"],
    "action_process": ["Concept", "action_process"],
    "measure_quantity": ["Concept", "measure_quantity"],
    "knowledge_linguistic": ["Concept", "knowledge_linguistic"],
    "other": ["Concept", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/concept.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class ConceptClassification(BaseModel):
    scratchpad: str = Field(
        description="step-by-step deduction of which of the leaf_categories the entity belongs to."
    )
    leaf_category: Literal[
        "abstract_concept",
        "attribute",
        "state",
        "action_process",
        "measure_quantity",
        "knowledge_linguistic",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=ConceptClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in CONCEPT_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

concept_entities = file.get("Concept", [])

print(f"Total Concept entities: {len(concept_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in concept_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Concept"]
        }

        entities[node_id] = entity

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)
    print(f"→ {leaf}")

    full_path = PARENT_MAP[leaf]
    existing_types = set(entity.get("types", []))

    # ----------------------------
    # add only if new
    # ----------------------------
    if set(full_path).issubset(existing_types):
        print("→ no new type to add")
        continue

    existing_types.update(full_path)
    entity["types"] = list(existing_types)

    # ----------------------------
    # save
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

    print("→ updated")

print("\nDONE.")

Total Concept entities: 99

Processing: पालक
→ other
→ updated

Processing: आशीर्वाद
→ action_process
→ updated

Processing: प्रकाशमय
→ attribute
→ updated

Processing: आशापूर्ण
→ attribute
→ updated

Processing: जितेन्द्रिय
→ attribute
→ updated

Processing: त्रिलोक के पाप
→ state
→ updated

Processing: कार्य-क्षमता
→ attribute
→ updated

Processing: विशेष
→ attribute
→ updated

Processing: अन्धकारमय
→ state
→ updated

Processing: आधुनिक शिक्षा
→ other
→ updated

Processing: मुख्य
→ attribute
→ updated

Processing: जिज्ञासा
→ abstract_concept
→ updated

Processing: अनर्थ
→ abstract_concept
→ updated

Processing: फलित
→ measure_quantity
→ updated

Processing: दुर्भिक्ष
→ state
→ updated

Processing: कडुआ
→ attribute
→ updated

Processing: पल्लवित
→ state
→ updated

Processing: संचार
→ knowledge_linguistic
→ updated

Processing: दानादि
→ action_process
→ updated

Processing: विनोदमयता
→ attribute
→ updated

Processing: सर्वमान्य
→ attribute
→ updated

Processing: ओजस्विता
→ attribute
→ 

In [13]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

# ============================================================
# GEOGRAPHY TREE
# ============================================================
GEO_LEAF_NODES = [
    "landform",
    "water_body",
    "vegetation_region",
    "atmospheric_region",
    "other"
]

PARENT_MAP = {
    "landform": ["Geographical_Feature", "landform"],
    "water_body": ["Geographical_Feature", "water_body"],
    "vegetation_region": ["Geographical_Feature", "vegetation_region"],
    "atmospheric_region": ["Geographical_Feature", "atmospheric_region"],
    "other": ["Geographical_Feature", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/geographical_feature.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class GeographyClassification(BaseModel):
    scratchpad: str = Field(
        description="step-by-step deduction of which of the leaf_categories the entity belongs to."
    )
    leaf_category: Literal[
        "landform",
        "water_body",
        "vegetation_region",
        "atmospheric_region",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 512,
        "temperature": 0.1,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=GeographyClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in GEO_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

geo_entities = file.get("Geographical_Feature", [])

print(f"Total Geographical_Feature entities: {len(geo_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in geo_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Geographical_Feature"]
        }

        entities[node_id] = entity

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)
    print(f"→ {leaf}")

    full_path = PARENT_MAP[leaf]
    existing_types = set(entity.get("types", []))

    # ----------------------------
    # add only if new
    # ----------------------------
    if set(full_path).issubset(existing_types):
        print("→ no new type to add")
        continue

    existing_types.update(full_path)
    entity["types"] = list(existing_types)

    # ----------------------------
    # save
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

    print("→ updated")

print("\nDONE.")

Total Geographical_Feature entities: 15

Processing: कुशों
→ vegetation_region
→ updated

Processing: वन
→ vegetation_region
→ updated

Processing: महानदी
→ water_body
→ updated

Processing: बिल
→ water_body
→ updated

Processing: गोदावरी
→ water_body
→ updated

Processing: जंगलों
→ vegetation_region
→ updated

Processing: परकोटे
→ landform
→ updated

Processing: चप्पे-चप्पे
→ other
→ updated

Processing: गोकुल
→ other
→ updated

Processing: कावेरी
→ water_body
→ updated

Processing: मरुस्थल
→ landform
→ updated

Processing: उपवन
→ vegetation_region
→ updated

Processing: पवित्र नदियों
→ water_body
→ updated

Processing: सिन्धु
→ water_body
→ updated

Processing: आधिभौतिक गंगाजी
→ water_body
→ updated

DONE.


In [15]:
import json
import os
from typing import Dict, List, Set

# ============================================================
# CONFIG
# ============================================================
ENTITY_FILE = "entity_info.json"  # Your existing entity repo
ENTITY_MAP_FILE = "entity_id_dict.json"  # Your existing entity name -> node_id mapping
CATEGORIES_FILE = "auto_categorized.json"

# ============================================================
# UTILS (reused from your original code)
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# ============================================================
# MAIN PROCESSING FUNCTION
# ============================================================
def merge_categories_into_repo():
    """
    Reads category dictionary (entity -> list of categories) and merges
    into existing entity repository.
    """
    
    # Load existing data
    entities = load_json(ENTITY_FILE, {})
    entity_map = load_json(ENTITY_MAP_FILE, {})
    
    # Load the new categories (the dictionary from your image)
    # Your image content shows a JSON dict, save it as a .json file
    categories_dict = load_json(CATEGORIES_FILE, {})
    
    print(f"Loaded {len(categories_dict)} entities from categories file")
    print(f"Existing entities in repo: {len(entities)}")
    
    stats = {
        "new_entities": 0,
        "updated_entities": 0,
        "skipped_entities": 0
    }
    
    # Process each entity from the categories dictionary
    for entity_name, new_categories in categories_dict.items():
        print(f"\nProcessing: {entity_name}")
        print(f"  Categories to add: {new_categories}")
        
        # Check if entity already exists in repo
        if entity_name in entity_map:
            # Entity exists - get its current types
            node_id = entity_map[entity_name]
            entity = entities[node_id]
            existing_types = set(entity.get("types", []))
            
            # Convert new_categories to set for comparison
            new_types_set = set(new_categories)
            
            # Find which categories are missing
            missing_types = new_types_set - existing_types
            
            if missing_types:
                print(f"  Missing categories: {missing_types}")
                # Add missing categories
                existing_types.update(missing_types)
                entity["types"] = list(existing_types)
                stats["updated_entities"] += 1
                print(f"  → Updated: added {len(missing_types)} new type(s)")
                
                # Save after each update (optional - can save at end for efficiency)
                atomic_save(entities, ENTITY_FILE)
                atomic_save(entity_map, ENTITY_MAP_FILE)
            else:
                print(f"  → All categories already present")
                stats["skipped_entities"] += 1
                
        else:
            # New entity - create it with all provided categories
            node_id = f"E_{len(entity_map):06d}"
            entity_map[entity_name] = node_id
            
            entity = {
                "node_id": node_id,
                "name": entity_name,
                "types": new_categories  # Use all categories from the dict
            }
            
            entities[node_id] = entity
            stats["new_entities"] += 1
            print(f"  → Created new entity with {len(new_categories)} type(s)")
            
            # Save after each new entity
            atomic_save(entities, ENTITY_FILE)
            atomic_save(entity_map, ENTITY_MAP_FILE)
    
    # Print summary
    print("\n" + "="*50)
    print("PROCESSING COMPLETE")
    print("="*50)
    print(f"✅ New entities added: {stats['new_entities']}")
    print(f"🔄 Existing entities updated: {stats['updated_entities']}")
    print(f"⏭️  Entities skipped (already complete): {stats['skipped_entities']}")
    print(f"\nTotal entities in repo now: {len(entities)}")


# ============================================================
# OPTIONAL: Function to validate and clean categories
# ============================================================
def validate_categories(categories_dict: Dict[str, List[str]]) -> Dict[str, List[str]]:
    """
    Ensures categories are properly formatted and removes duplicates.
    """
    cleaned = {}
    
    for entity, categories in categories_dict.items():
        # Remove any None or empty strings
        valid_cats = [c for c in categories if c and isinstance(c, str)]
        # Remove duplicates while preserving order
        seen = set()
        unique_cats = []
        for cat in valid_cats:
            if cat not in seen:
                seen.add(cat)
                unique_cats.append(cat)
        
        if unique_cats:
            cleaned[entity] = unique_cats
        else:
            print(f"Warning: Entity '{entity}' has no valid categories, skipping")
    
    return cleaned


# ============================================================
# ALTERNATIVE: Batch save version (more efficient for large files)
# ============================================================
def merge_categories_in_batch():
    """
    Same as above but saves only once at the end (better for large files).
    """
    entities = load_json(ENTITY_FILE, {})
    entity_map = load_json(ENTITY_MAP_FILE, {})
    categories_dict = load_json(CATEGORIES_FILE, {})
    
    print(f"Loaded {len(categories_dict)} entities from categories file")
    print(f"Existing entities in repo: {len(entities)}")
    
    # Optional: clean categories first
    categories_dict = validate_categories(categories_dict)
    
    stats = {"new": 0, "updated": 0, "skipped": 0}
    
    # Process all entities in memory first
    for entity_name, new_categories in categories_dict.items():
        if entity_name in entity_map:
            # Existing entity
            node_id = entity_map[entity_name]
            entity = entities[node_id]
            existing_types = set(entity.get("types", []))
            new_types_set = set(new_categories)
            missing_types = new_types_set - existing_types
            
            if missing_types:
                existing_types.update(missing_types)
                entity["types"] = list(existing_types)
                stats["updated"] += 1
                print(f"✓ Updated: {entity_name} (+{len(missing_types)} types)")
            else:
                stats["skipped"] += 1
                print(f"→ Skipped: {entity_name} (already complete)")
        else:
            # New entity
            node_id = f"E_{len(entity_map):06d}"
            entity_map[entity_name] = node_id
            entities[node_id] = {
                "node_id": node_id,
                "name": entity_name,
                "types": new_categories
            }
            stats["new"] += 1
            print(f"+ Created: {entity_name} ({len(new_categories)} types)")
    
    # Save everything once
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)
    
    print("\n" + "="*50)
    print(f"✅ New: {stats['new']} | 🔄 Updated: {stats['updated']} | ⏭️ Skipped: {stats['skipped']}")
    print(f"Total entities now: {len(entities)}")


# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    # Choose which method to use:
    
    # Method 1: Save after each entity (safer for long-running processes)
    # merge_categories_into_repo()
    
    # Method 2: Batch save at the end (more efficient)
    merge_categories_in_batch()

Loaded 1092 entities from categories file
Existing entities in repo: 1492
+ Created: कालावयवों (4 types)
+ Created: द्वादशी व्रत (2 types)
+ Created: जन्म-महोत्सव (2 types)
+ Created: विद्वान व्यक्ति (3 types)
+ Created: मृत शरीर (1 types)
+ Created: शस्त्र (1 types)
+ Created: षोडशो-पचार (2 types)
+ Created: लड़के (3 types)
+ Created: पोली गाय का दूध (1 types)
+ Created: समग्र जगत् (5 types)
+ Created: श्रेष्ठ (2 types)
+ Created: गायन (2 types)
+ Created: निर्धन वैष्णव (1 types)
+ Created: कार्तिक कृष्णा चतुर्दशी (3 types)
+ Created: मल्लविद्याविदों (3 types)
+ Created: ॐ यशोदायै नमः (1 types)
+ Created: अनुचरों (2 types)
+ Created: शेषनाग (2 types)
+ Created: स्कन्द‌पुराण (1 types)
+ Created: भद्रा-रहित समय (2 types)
+ Created: शोभा (2 types)
+ Created: नाग पूजा (2 types)
+ Created: ग्रन्थ (2 types)
+ Created: निर्णायक (4 types)
+ Created: मोदक (1 types)
+ Created: कान की पीडा (5 types)
+ Created: विश्वम्भर (5 types)
+ Created: निम्बार्कसम्प्रदायी (1 types)
+ Created: अगले वर्ष (2 t